<a href="https://colab.research.google.com/github/JC9427/JC-programming/blob/main/HW3_%E5%BE%85%E8%BE%A6%E6%B8%85%E5%96%AE%E8%88%87%E7%95%AA%E8%8C%84%E9%90%98%E7%B4%80%E9%8C%84_Gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 功能
- 任務下拉選單
- 番茄鐘換算
- 爬蟲
- 倒數計時

Googlesheet：https://docs.google.com/spreadsheets/d/1IvgBEfbQ7hWF1xYT26xdbKxP3GhlmDsH2liGxv2SjqI/edit?usp=sharing

In [1]:
!pip -q install gspread gspread_dataframe google-auth google-auth-oauthlib google-auth-httplib2 \
               gradio pandas beautifulsoup4 google-generativeai python-dateutil openai

In [2]:
from urllib.parse import urljoin
import os, time, uuid, re, json, datetime
from datetime import datetime as dt, timedelta
from dateutil.tz import gettz
import pandas as pd
import gradio as gr
import requests
from bs4 import BeautifulSoup

import google.generativeai as genai

# Google Auth & Sheets
from google.colab import auth
import gspread
from gspread_dataframe import set_with_dataframe, get_as_dataframe
from google.auth.transport.requests import Request
from google.oauth2 import service_account
from google.auth import default

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
#讓 Colab 取得你的 Google 帳號授權，這樣程式才能讀取和寫入你的 Google Sheet。

from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

In [4]:
from google.colab import userdata
import openai

# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gpt')

# 使用獲取的金鑰配置 OpenAI API
openai.api_key = api_key

In [5]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1IvgBEfbQ7hWF1xYT26xdbKxP3GhlmDsH2liGxv2SjqI/edit?usp=sharing"
WORKSHEET_NAME = "番茄鐘"
TIMEZONE = "Asia/Taipei"

In [6]:
import pandas as pd

# 在 google 工作表載入 gsheets
gsheets = gc.open_by_url(SHEET_URL)

# 從 gsheets 的番茄鐘工作表載入資料
sh = gsheets.worksheet(WORKSHEET_NAME).get_all_values()

# 將資料載入 DataFrame
df = pd.DataFrame(sh[1:], columns=sh[0])

# 取得最前面的5筆資料
df.head()

,id,task,status,priority,est_min,start_time,end_time,actual_min,pomodoros,due_date,labels,notes,created_at,updated_at,completed_at,planned_for
0,fe006ea1,剪頭髮,todo,H,60,,,0,0,2026-04-30,,,2026-04-30T10:50:58.711401+08:00,2026-04-30T10:50:58.711401+08:00,,today
1,a5de0641,看電影,todo,M,120,,,0,0,2026-05-10,,,2026-05-04T20:24:18.424431+08:00,2026-05-11T19:37:08.634585+08:00,,
2,d18251bc,2026赤峰草地音樂會【經典凱西大集合】,todo,L,25,,,0,0,,"from:crawler,活動",來源：https://www.accupass.com/?area=north\n選擇器：a...,2026-05-11T19:18:33.216252+08:00,2026-05-11T19:18:33.216252+08:00,,


In [7]:
def ensure_spreadsheet(name):
    try:
        sh = gc.open(name)
    except gspread.SpreadsheetNotFound:
        sh = gc.create(name)
    return sh

sh = ensure_spreadsheet(WORKSHEET_NAME)

In [8]:
# =========================
# Google Sheet 設定
# =========================

gsheets = gc.open_by_url(SHEET_URL)

def ensure_worksheet(spreadsheet, title, header):
    try:
        ws = spreadsheet.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = spreadsheet.add_worksheet(
            title=title,
            rows="1000",
            cols=str(len(header) + 5)
        )
        ws.update([header])

    data = ws.get_all_values()

    if not data:
        ws.update([header])
    else:
        current_header = data[0]
        if current_header != header:
            ws.clear()
            ws.update([header])

    return ws


TASKS_HEADER = [
    "id", "task", "status", "priority", "est_min",
    "start_time", "end_time", "actual_min", "pomodoros",
    "due_date", "labels", "notes",
    "created_at", "updated_at", "completed_at", "planned_for"
]

LOGS_HEADER = [
    "log_id", "task_id", "phase", "start_ts",
    "end_ts", "minutes", "cycles", "note"
]

CLIPS_HEADER = [
    "clip_id", "url", "selector", "text",
    "href", "created_at", "added_to_task"
]

# 這裡會寫入你的 SHEET_URL 這份 Google Sheet 裡面的分頁
ws_tasks = ensure_worksheet(gsheets, WORKSHEET_NAME, TASKS_HEADER)
ws_logs  = ensure_worksheet(gsheets, "pomodoro_logs", LOGS_HEADER)
ws_clips = ensure_worksheet(gsheets, "web_clips", CLIPS_HEADER)


# =========================
# 基本工具
# =========================

def tznow():
    return dt.now(gettz(TIMEZONE))


def read_df(ws, header):
    data = ws.get_all_values()

    if not data or len(data) <= 1:
        return pd.DataFrame(columns=header)

    df = pd.DataFrame(data[1:], columns=data[0])
    df = df.fillna("")

    for c in header:
        if c not in df.columns:
            df[c] = ""

    df = df[header]

    for col in ["est_min", "actual_min", "pomodoros"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

    return df


def write_df(ws, df, header):
    ws.clear()

    if df.empty:
        ws.update([header])
        return

    df_out = df.copy()

    for c in header:
        if c not in df_out.columns:
            df_out[c] = ""

    df_out = df_out[header].fillna("")

    for c in df_out.columns:
        df_out[c] = df_out[c].astype(str)

    ws.update([header] + df_out.values.tolist())


def refresh_all():
    return (
        read_df(ws_tasks, TASKS_HEADER).copy(),
        read_df(ws_logs, LOGS_HEADER).copy(),
        read_df(ws_clips, CLIPS_HEADER).copy()
    )


tasks_df, logs_df, clips_df = refresh_all()


# =========================
# 任務下拉選單
# =========================

def list_task_choices():
    global tasks_df

    if tasks_df.empty:
        return []

    df = tasks_df[tasks_df["status"] != "done"].copy()

    if df.empty:
        return []

    def row_label(r):
        return f"[{r['status']}] (P:{r['priority']}) {r['task']}"

    return [(row_label(r), r["id"]) for _, r in df.iterrows()]


def get_task_cycles(task_id):
    global tasks_df

    if not task_id:
        return gr.update(value=1)

    row = tasks_df[tasks_df["id"] == task_id]

    if row.empty:
        return gr.update(value=1)

    est_min = int(row.iloc[0]["est_min"])
    cycles = max(1, int((est_min + 24) // 25))

    return gr.update(value=cycles)


# =========================
# 新增任務：會寫入 Google Sheet
# =========================

def add_task(task, priority, est_min, due_date, labels, notes, planned_for):
    global tasks_df

    if not task or not str(task).strip():
        return (
            "⚠️ 請輸入任務名稱",
            tasks_df,
            gr.update(choices=list_task_choices()),
            gr.update(choices=list_task_choices())
        )

    _now = tznow().isoformat()
    task_id = str(uuid.uuid4())[:8]

    new_task = {
        "id": task_id,
        "task": str(task).strip(),
        "status": "todo",
        "priority": priority or "M",
        "est_min": int(est_min) if est_min else 25,
        "start_time": "",
        "end_time": "",
        "actual_min": 0,
        "pomodoros": 0,
        "due_date": due_date or "",
        "labels": labels or "",
        "notes": notes or "",
        "created_at": _now,
        "updated_at": _now,
        "completed_at": "",
        "planned_for": planned_for or ""
    }

    tasks_df = pd.concat(
        [tasks_df, pd.DataFrame([new_task])],
        ignore_index=True
    )

    write_df(ws_tasks, tasks_df, TASKS_HEADER)

    return (
        f"✅ 已新增任務，並寫入 Google Sheet：{new_task['task']}",
        tasks_df,
        gr.update(choices=list_task_choices(), value=task_id),
        gr.update(choices=list_task_choices(), value=task_id)
    )


# =========================
# 更新任務狀態
# =========================

def update_task_status(task_id, new_status):
    global tasks_df

    if not task_id:
        return (
            "⚠️ 請先選擇任務",
            tasks_df,
            gr.update(choices=list_task_choices()),
            gr.update(choices=list_task_choices())
        )

    idx = tasks_df.index[tasks_df["id"] == task_id]

    if len(idx) == 0:
        return (
            "⚠️ 找不到任務",
            tasks_df,
            gr.update(choices=list_task_choices()),
            gr.update(choices=list_task_choices())
        )

    i = idx[0]

    tasks_df.loc[i, "status"] = new_status
    tasks_df.loc[i, "updated_at"] = tznow().isoformat()

    if new_status == "done":
        tasks_df.loc[i, "completed_at"] = tznow().isoformat()

    write_df(ws_tasks, tasks_df, TASKS_HEADER)

    return (
        "✅ 狀態已更新，並同步到 Google Sheet",
        tasks_df,
        gr.update(choices=list_task_choices(), value=None),
        gr.update(choices=list_task_choices(), value=None)
    )


def mark_done(task_id):
    return update_task_status(task_id, "done")


# =========================
# 番茄鐘紀錄
# =========================

def recalc_task_actuals(task_id):
    global tasks_df, logs_df

    work_logs = logs_df[
        (logs_df["task_id"] == task_id) &
        (logs_df["phase"] == "work")
    ]

    total_min = work_logs["minutes"].astype(float).sum() if not work_logs.empty else 0
    pomos = int(total_min // 25)

    idx = tasks_df.index[tasks_df["id"] == task_id]

    if len(idx) == 0:
        return

    i = idx[0]
    tasks_df.loc[i, "actual_min"] = int(total_min)
    tasks_df.loc[i, "pomodoros"] = pomos
    tasks_df.loc[i, "updated_at"] = tznow().isoformat()


_active_sessions = {}

def format_seconds(seconds):
    seconds = max(0, int(seconds))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60

    if h > 0:
        return f"{h:02d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"


def update_timer_display(task_id):
    if not task_id or task_id not in _active_sessions:
        return "⏱️ 尚未開始計時"

    sess = _active_sessions[task_id]
    start = pd.to_datetime(sess["start_ts"])
    now = tznow()

    elapsed_seconds = (now - start).total_seconds()

    phase = sess["phase"]
    cycles = int(sess["cycles"])

    if phase == "work":
        target_seconds = 25 * 60 * cycles
        phase_name = "工作"
    else:
        target_seconds = 15 * 60
        phase_name = "休息"

    remaining_seconds = target_seconds - elapsed_seconds

    return (
        f"⏱️ 目前階段：{phase_name}\n\n"
        f"已經過：{format_seconds(elapsed_seconds)}\n\n"
        f"剩餘時間：{format_seconds(remaining_seconds)}"
    )


def start_phase(task_id, phase, cycles):
    if not task_id:
        return "⚠️ 請先選擇任務"

    _active_sessions[task_id] = {
        "phase": phase,
        "start_ts": tznow().isoformat(),
        "cycles": int(cycles) if cycles else 1
    }

    if phase == "work":
        return "▶️ 開始工作（建議 25 分鐘）"
    else:
        return "🍵 開始休息（建議 5–15 分鐘）"


def end_phase(task_id, note):
    global logs_df, tasks_df

    if not task_id:
        return "⚠️ 請先選擇任務"

    if task_id not in _active_sessions:
        return "⚠️ 尚未開始任何階段"

    sess = _active_sessions.pop(task_id)

    start = pd.to_datetime(sess["start_ts"])
    end = tznow()
    minutes = round((end - start).total_seconds() / 60.0, 2)

    new_log = {
        "log_id": str(uuid.uuid4())[:8],
        "task_id": task_id,
        "phase": sess["phase"],
        "start_ts": start.isoformat(),
        "end_ts": end.isoformat(),
        "minutes": minutes,
        "cycles": int(sess["cycles"]),
        "note": note or ""
    }

    logs_df = pd.concat(
        [logs_df, pd.DataFrame([new_log])],
        ignore_index=True
    )

    write_df(ws_logs, logs_df, LOGS_HEADER)

    if sess["phase"] == "work":
        recalc_task_actuals(task_id)
        write_df(ws_tasks, tasks_df, TASKS_HEADER)

    return f"⏹️ 已結束：{sess['phase']}，紀錄 {minutes} 分鐘"


# =========================
# AI 今日計畫
# =========================

def generate_today_plan():
    global tasks_df

    today = tznow().date().isoformat()

    cand = tasks_df[
        (
            (tasks_df["due_date"] == today) |
            (tasks_df["planned_for"].str.lower() == "today")
        ) &
        (tasks_df["status"] != "done")
    ].copy()

    if cand.empty:
        return "📭 今天沒有標記的任務。請把任務的 due_date 設為今天，或 planned_for 設為 today。"

    pr_order = {"H": 0, "M": 1, "L": 2}
    cand["p_ord"] = cand["priority"].map(pr_order).fillna(3)
    cand = cand.sort_values(["p_ord", "est_min"], ascending=[True, True])

    buckets = {
        "morning": [],
        "afternoon": [],
        "evening": []
    }

    for i, (_, r) in enumerate(cand.iterrows()):
        if i % 3 == 0:
            buckets["morning"].append(r)
        elif i % 3 == 1:
            buckets["afternoon"].append(r)
        else:
            buckets["evening"].append(r)

    def sec_md(name, rows):
        if not rows:
            return f"### {name.title()}\n（無）\n"

        lines = [f"### {name.title()}"]

        for r in rows:
            lines.append(
                f"- [{r['id']}] {r['task']}（預估 {int(r['est_min'])} 分，P:{r['priority']}）"
            )

        return "\n".join(lines) + "\n"

    return (
        sec_md("morning", buckets["morning"]) + "\n" +
        sec_md("afternoon", buckets["afternoon"]) + "\n" +
        sec_md("evening", buckets["evening"])
    )


# =========================
# 今日完成率
# =========================

def today_summary():
    global tasks_df

    today = tznow().date().isoformat()

    planned = tasks_df[
        (tasks_df["due_date"] == today) |
        (tasks_df["planned_for"].str.lower() == "today")
    ]

    done = planned[planned["status"] == "done"]

    total = len(planned)
    done_n = len(done)
    rate = (done_n / total * 100) if total > 0 else 0

    return f"📅 今日計畫任務：{total}；✅ 完成：{done_n}；📈 完成率：{rate:.1f}%"


# =========================
# 爬蟲功能
# =========================

def clean_text(text):
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def match_keyword(text, href, keyword):
    """
    如果沒有填關鍵字：全部保留
    如果有填關鍵字：只保留 text 或 href 有包含關鍵字的資料
    """
    if not keyword or not str(keyword).strip():
        return True

    keyword = str(keyword).strip().lower()
    text = str(text).lower()
    href = str(href).lower()

    return keyword in text or keyword in href


def crawl(url, selector, keyword, mode, limit):
    if not url or not str(url).strip():
        return pd.DataFrame(columns=CLIPS_HEADER), "⚠️ 請先輸入目標 URL"

    try:
        resp = requests.get(
            url,
            timeout=15,
            headers={
                "User-Agent": "Mozilla/5.0"
            }
        )

        # 如果網站回傳 403、404、500 這類錯誤，這裡會進入 except
        resp.raise_for_status()

    except requests.exceptions.HTTPError as e:
        return (
            pd.DataFrame(columns=CLIPS_HEADER),
            f"⚠️ 網站讀取失敗，可能不是公開頁面或伺服器拒絕存取：{e}"
        )

    except requests.exceptions.Timeout:
        return (
            pd.DataFrame(columns=CLIPS_HEADER),
            "⚠️ 網站讀取逾時，可能網站太慢或不允許爬取"
        )

    except requests.exceptions.RequestException as e:
        return (
            pd.DataFrame(columns=CLIPS_HEADER),
            f"⚠️ 無法連線到這個網站：{e}"
        )

    try:
        soup = BeautifulSoup(resp.text, "html.parser")
    except Exception as e:
        return (
            pd.DataFrame(columns=CLIPS_HEADER),
            f"⚠️ 網頁解析失敗：{e}"
        )

    # 如果 CSS Selector 空白，就預設抓所有超連結
    if not selector or not str(selector).strip():
        selector = "a"

    try:
        nodes = soup.select(selector)
    except Exception as e:
        return (
            pd.DataFrame(columns=CLIPS_HEADER),
            f"⚠️ CSS Selector 格式錯誤：{e}"
        )

    rows = []

    for n in nodes:
        text = ""
        href = ""

        # 抓文字
        if mode in ("text", "both"):
            text = clean_text(n.get_text(" ", strip=True))

        # 抓連結
        if mode in ("href", "both"):
            if n.name == "a":
                href = n.get("href", "")
            else:
                a_tag = n.find("a")
                if a_tag:
                    href = a_tag.get("href", "")

            if href:
                href = urljoin(url, href)

        # 如果選 text 模式，也還是幫你抓 href，方便之後轉成任務備註
        if mode == "text":
            if n.name == "a":
                href = n.get("href", "")
            else:
                a_tag = n.find("a")
                if a_tag:
                    href = a_tag.get("href", "")

            if href:
                href = urljoin(url, href)

        # 如果選 href 模式，也盡量保留文字
        if mode == "href":
            text = clean_text(n.get_text(" ", strip=True))

        # 文字和連結都沒有，就跳過
        if not text and not href:
            continue

        # 過濾太短、沒有意義的文字
        if len(text) <= 1 and not href:
            continue

        # 關鍵字篩選
        if not match_keyword(text, href, keyword):
            continue

        rows.append({
            "clip_id": str(uuid.uuid4())[:8],
            "url": url,
            "selector": selector,
            "text": text,
            "href": href,
            "created_at": tznow().isoformat(),
            "added_to_task": ""
        })

        if limit and len(rows) >= int(limit):
            break

    df = pd.DataFrame(rows, columns=CLIPS_HEADER)

    if df.empty:
        return df, "⚠️ 沒有抓到符合條件的資料。可能是：關鍵字不符合、CSS Selector 不對、或內容是 JavaScript 動態載入。"

    return df, f"✅ 成功擷取 {len(df)} 筆資料"

    # =========================
    # 一般網站模式
    # =========================

    if not selector or not str(selector).strip():
        selector = "a"

    nodes = soup.select(selector)

    for n in nodes:

        text = clean_event_text(n.get_text(" ", strip=True))

        href = ""

        if n.name == "a":
            href = n.get("href", "")
        else:
            a_tag = n.find("a")
            if a_tag:
                href = a_tag.get("href", "")

        if href:
            href = urljoin(url, href)

        if not text and not href:
            continue

        if is_bad_accupass_text(text):
            continue

        if not match_keyword(text, keyword):
            continue

        rows.append({
            "clip_id": str(uuid.uuid4())[:8],
            "url": url,
            "selector": selector,
            "text": text,
            "href": href,
            "created_at": tznow().isoformat(),
            "added_to_task": ""
        })

        if limit and len(rows) >= int(limit):
            break

    df = pd.DataFrame(rows, columns=CLIPS_HEADER)

    return df, f"✅ 擷取 {len(df)} 筆符合關鍵字的資料"

        # 如果 Accupass 專用方法抓不到，會往下走一般模式

    # =========================
    # 一般網站模式
    # =========================

    if not selector or not str(selector).strip():
        selector = "a"

    nodes = soup.select(selector)

    for n in nodes[:int(limit) if limit else 20]:

        text = clean_event_text(n.get_text(" ", strip=True))

        href = ""

        if n.name == "a":
            href = n.get("href", "")
        else:
            a_tag = n.find("a")
            if a_tag:
                href = a_tag.get("href", "")

        if href:
            href = urljoin(url, href)

        if not text and not href:
            continue

        # 一般模式也排除太短、太像分類選單的文字
        if is_bad_accupass_text(text):
            continue

        rows.append({
            "clip_id": str(uuid.uuid4())[:8],
            "url": url,
            "selector": selector,
            "text": text,
            "href": href,
            "created_at": tznow().isoformat(),
            "added_to_task": ""
        })

    df = pd.DataFrame(rows, columns=CLIPS_HEADER)

    return df, f"✅ 擷取 {len(df)} 筆資料"


def add_clips_as_tasks(clip_ids, default_priority, est_min):
    global clips_df, tasks_df

    if not clip_ids:
        return (
            "⚠️ 請先輸入要加入的 clip_id",
            clips_df,
            tasks_df,
            gr.update(choices=list_task_choices()),
            gr.update(choices=list_task_choices())
        )

    sel = clips_df[clips_df["clip_id"].isin(clip_ids)]

    if sel.empty:
        return (
            "⚠️ 找不到對應的 clip_id",
            clips_df,
            tasks_df,
            gr.update(choices=list_task_choices()),
            gr.update(choices=list_task_choices())
        )

    _now = tznow().isoformat()
    new_tasks = []

    for _, r in sel.iterrows():
        title = r["text"] or r["href"] or "（未命名）"
        note = f"來源：{r['url']}\n選擇器：{r['selector']}\n活動連結：{r['href']}"

        new_tasks.append({
            "id": str(uuid.uuid4())[:8],
            "task": title[:120],
            "status": "todo",
            "priority": default_priority or "M",
            "est_min": int(est_min) if est_min else 25,
            "start_time": "",
            "end_time": "",
            "actual_min": 0,
            "pomodoros": 0,
            "due_date": "",
            "labels": "from:crawler,活動",
            "notes": note,
            "created_at": _now,
            "updated_at": _now,
            "completed_at": "",
            "planned_for": ""
        })

    tasks_df = pd.concat(
        [tasks_df, pd.DataFrame(new_tasks)],
        ignore_index=True
    )

    clips_df.loc[clips_df["clip_id"].isin(clip_ids), "added_to_task"] = "yes"

    write_df(ws_tasks, tasks_df, TASKS_HEADER)
    write_df(ws_clips, clips_df, CLIPS_HEADER)

    return (
        f"✅ 已加入 {len(new_tasks)} 項為任務，並寫入 Google Sheet",
        clips_df,
        tasks_df,
        gr.update(choices=list_task_choices()),
        gr.update(choices=list_task_choices())
    )


# =========================
# 重新整理
# =========================

def _refresh():
    global tasks_df, logs_df, clips_df

    tasks_df, logs_df, clips_df = refresh_all()
    task_choices = list_task_choices()

    return (
        tasks_df,
        logs_df,
        clips_df,
        gr.update(choices=task_choices),
        gr.update(choices=task_choices),
        today_summary()
    )


# =========================
# Gradio 介面
# =========================

with gr.Blocks(title="待辦清單＋番茄鐘＋AI 計畫") as demo:
    gr.Markdown("# ✅ 待辦清單與番茄鐘（Google Sheet 同步版）")

    with gr.Row():
        btn_refresh = gr.Button("🔄 重新整理（Sheet → App）")
        out_summary = gr.Markdown(today_summary())

    with gr.Tab("Tasks"):
        with gr.Row():
            with gr.Column(scale=2):
                task = gr.Textbox(label="任務名稱", placeholder="例如：寫作業 / 看書 / 做報告")
                priority = gr.Dropdown(["H", "M", "L"], value="M", label="優先級")
                est_min = gr.Number(value=25, label="預估時間（分鐘）", precision=0)
                due_date = gr.Textbox(label="到期日（YYYY-MM-DD，可空白）")
                labels = gr.Textbox(label="標籤（例如：學校, 作業, 考試）")
                notes = gr.Textbox(label="備註（可空白）")
                planned_for = gr.Dropdown(["", "today", "tomorrow"], value="", label="規劃歸屬")

                btn_add = gr.Button("➕ 新增任務到 Google Sheet")
                msg_add = gr.Markdown()

            with gr.Column(scale=3):
                grid_tasks = gr.Dataframe(
                    value=tasks_df,
                    label="任務清單（Google Sheet 資料）",
                    interactive=False
                )

        with gr.Row():
            task_choice = gr.Dropdown(
                choices=list_task_choices(),
                label="選取任務（用於更新）",
                value=None
            )
            new_status = gr.Dropdown(
                ["todo", "in-progress", "done"],
                value="in-progress",
                label="更新狀態"
            )
            btn_update = gr.Button("✏️ 更新狀態")
            btn_done = gr.Button("✅ 直接標記完成")
            msg_update = gr.Markdown()

    with gr.Tab("Pomodoro"):
        with gr.Row():
            sel_task = gr.Dropdown(
                choices=list_task_choices(),
                label="選擇任務",
                value=None
            )
            cycles = gr.Number(value=1, precision=0, label="番茄數（依預估時間自動更新）")

        timer_display = gr.Markdown(
            value="⏱️ 尚未開始計時",
            label="計時器"
        )

        timer = gr.Timer(1)

        sel_task.change(
            get_task_cycles,
            inputs=[sel_task],
            outputs=[cycles]
        )

        with gr.Row():
            btn_start_work = gr.Button("▶️ 開始工作")
            note_work = gr.Textbox(label="工作備註（可空白）")
            btn_end_work = gr.Button("⏹️ 結束工作並記錄")

        with gr.Row():
            btn_start_break = gr.Button("🍵 開始休息")
            note_break = gr.Textbox(label="休息備註（可空白）")
            btn_end_break = gr.Button("⏹️ 結束休息並記錄")

        msg_pomo = gr.Markdown()

        grid_logs = gr.Dataframe(
            value=logs_df,
            label="番茄鐘紀錄",
            interactive=False
        )

    with gr.Tab("AI Plan"):
        gr.Markdown("根據今天的任務，排成 morning / afternoon / evening。")
        btn_plan = gr.Button("🧠 產生今日計畫")
        out_plan = gr.Markdown()

    with gr.Tab("Crawler"):
        url = gr.Textbox(
            label="目標 URL",
            placeholder="貼上公開網頁網址，例如：https://example.com/events"
        )

        selector = gr.Textbox(
            label="CSS Selector",
            placeholder=""
        )

        keyword = gr.Textbox(
            label="搜尋關鍵字",
            placeholder="例如：音樂會、講座、設計、攝影、比賽"
        )

        mode = gr.Radio(
            ["text", "href", "both"],
            value="both",
            label="擷取內容"
        )

        limit = gr.Number(
            value=20,
            precision=0,
            label="最多擷取幾筆"
        )

        btn_crawl = gr.Button("🕷️ 開始擷取")
        msg_crawl = gr.Markdown()

        grid_clips = gr.Dataframe(
            value=clips_df,
            label="擷取結果",
            interactive=True
        )

        clip_ids = gr.Textbox(label="要加入任務的 clip_id（多個以逗號分隔）")
        default_priority = gr.Dropdown(["H", "M", "L"], value="L", label="新增任務優先級")
        clip_est = gr.Number(value=25, precision=0, label="新增任務預估分鐘")
        btn_add_clips = gr.Button("➕ 將擷取項目加入為任務")
        msg_add_clips = gr.Markdown()

    with gr.Tab("Summary"):
        btn_summary = gr.Button("📊 重新計算今日完成率")
        out_summary2 = gr.Markdown()

    btn_refresh.click(
        _refresh,
        outputs=[
            grid_tasks,
            grid_logs,
            grid_clips,
            task_choice,
            sel_task,
            out_summary
        ]
    )

    btn_add.click(
        add_task,
        inputs=[
            task,
            priority,
            est_min,
            due_date,
            labels,
            notes,
            planned_for
        ],
        outputs=[
            msg_add,
            grid_tasks,
            task_choice,
            sel_task
        ]
    )

    btn_update.click(
        update_task_status,
        inputs=[task_choice, new_status],
        outputs=[
            msg_update,
            grid_tasks,
            task_choice,
            sel_task
        ]
    )

    btn_done.click(
        mark_done,
        inputs=[task_choice],
        outputs=[
            msg_update,
            grid_tasks,
            task_choice,
            sel_task
        ]
    )

    btn_start_work.click(
        start_phase,
        inputs=[sel_task, gr.State("work"), cycles],
        outputs=[msg_pomo]
    )

    btn_end_work.click(
        end_phase,
        inputs=[sel_task, note_work],
        outputs=[msg_pomo]
    )

    btn_start_break.click(
        start_phase,
        inputs=[sel_task, gr.State("break"), cycles],
        outputs=[msg_pomo]
    )

    btn_end_break.click(
        end_phase,
        inputs=[sel_task, note_break],
        outputs=[msg_pomo]
    )

    timer.tick(
        update_timer_display,
        inputs=[sel_task],
        outputs=[timer_display]
    )

    btn_plan.click(
        generate_today_plan,
        outputs=[out_plan]
    )

    def _crawl_and_save(u, s, k, m, l):
        global clips_df

        df, msg = crawl(u, s, k, m, l)

        if not df.empty:
            clips_df = pd.concat([clips_df, df], ignore_index=True)
            write_df(ws_clips, clips_df, CLIPS_HEADER)

        return msg, clips_df

    btn_crawl.click(
        _crawl_and_save,
        inputs=[url, selector, keyword, mode, limit],
        outputs=[msg_crawl, grid_clips]
    )

    def _add_clips(clip_ids_str, pr, est):
        ids = [
            c.strip()
            for c in (clip_ids_str or "").split(",")
            if c.strip()
        ]

        return add_clips_as_tasks(ids, pr, est)

    btn_add_clips.click(
        _add_clips,
        inputs=[clip_ids, default_priority, clip_est],
        outputs=[
            msg_add_clips,
            grid_clips,
            grid_tasks,
            task_choice,
            sel_task
        ]
    )

    btn_summary.click(
        today_summary,
        outputs=[out_summary2]
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://22da8683165e85fb5d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
